# M2.3: European market design — timeline, bidding zones and market coupling

## Objectives

At the end of this notebook you should be able to:

- describe the sequence of European electricity markets — forward, **day-ahead**, **intraday**, balancing — and what each is for
- explain **bidding zones** and where Croatia sits in the coupled European market
- clear several zones together as one welfare-maximisation LP (**market coupling**), and see why prices **converge** across free borders and **split** across congested ones — checked against PyPSA
- compute **congestion income** and show that welfare = consumer surplus + producer surplus + congestion income
- explain why simple transfer capacities ignore physics, and what **flow-based** coupling adds
- load real **day-ahead prices for HR, SI and HU** from ENTSO-E and measure how often they converge

M2.1 cleared one market. Europe clears dozens of bidding zones together every day. This lesson builds that idea up with three zones — Slovenia, Croatia, Hungary — first synthetic, then real.

In [1]:
import logging
import os
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### The market timeline

Electricity for a given quarter-hour is traded several times, from years ahead until minutes before delivery:

| Market | When | What it does |
|---|---|---|
| **Forward / futures** | years to days ahead | hedges price risk; mostly financial |
| **Day-ahead** (SDAC) | one auction per day, gate closure **12:00 CET** the day before | sets the reference price for every bidding zone and allocates cross-border capacity implicitly |
| **Intraday** (SIDC) | continuous trading plus intraday auctions, until shortly before delivery | corrects forecast errors (wind, solar, outages) after the day-ahead result |
| **Balancing** | in real time, run by the TSO | activates reserves to keep supply and demand equal second by second; imbalances are settled afterwards |

The **Single Day-Ahead Coupling** (SDAC) clears all participating zones together with one algorithm (EUPHEMIA). Since delivery day **1 October 2025** it clears in **15-minute** market time units instead of hours ([SDAC](https://www.entsoe.eu/network_codes/cacm/implementation/sdac/)). That matters for data: day-ahead prices from October 2025 onwards come at 15-minute resolution.

### Croatia in the coupled market

Croatia is **one bidding zone** (one price), with CROPEX as its power exchange and HOPS as its TSO. Its day-ahead market is coupled to SDAC over the **Croatian–Slovenian** border and, since **June 2022**, over the **Croatian–Hungarian** border, which came in with the flow-based capacity calculation of the Core region ([CROPEX](https://www.cropex.hr/en/market-data/general.html)). The borders with non-EU neighbours (Bosnia and Herzegovina, Serbia) are not part of SDAC; capacity there is allocated separately. Croatia's prices can still be extreme: CROPEX reports a lowest day-ahead price of about −151 €/MWh in 2025 — the negative prices of M2.2 are real.

### Three zones, one auction

Three synthetic zones, sized loosely like Slovenia, Croatia and Hungary, each with its own order book (M2.1). Croatia has hydro and wind but relies on expensive gas; Slovenia and Hungary have more cheap nuclear and solar:

In [2]:
ZONES = ["SI", "HR", "HU"]
offers = pd.DataFrame([
    ("SI", "nuclear", 10, 700), ("SI", "hydro", 5, 500), ("SI", "coal", 95, 500), ("SI", "gas", 115, 300),
    ("HR", "hydro", 5, 900), ("HR", "wind", 0, 500), ("HR", "gas", 120, 700), ("HR", "ocgt", 160, 400),
    ("HU", "nuclear", 12, 1900), ("HU", "solar", 0, 800), ("HU", "gas", 100, 1500), ("HU", "ocgt", 150, 500),
], columns=["zone", "order", "price", "qty"])                         # EUR/MWh, MW
bids = pd.DataFrame([
    ("SI", "load", 3000, 1300), ("HR", "load", 3000, 2400), ("HR", "industry", 130, 150),
    ("HU", "load", 3000, 3500), ("HU", "storage", 60, 300),
], columns=["zone", "order", "price", "qty"])
print(offers.groupby("zone").qty.sum().rename("offered_MW").to_frame().join(bids.groupby("zone").qty.sum().rename("bid_MW")))

      offered_MW  bid_MW
zone                    
HR          2500    2550
HU          4700    3800
SI          2000    1300


Market coupling is M2.1's welfare LP with one balance per zone and a **flow variable per border**, limited by the border's transfer capacity (ATC):

**maximise** Σ bids p·d − Σ offers p·s  **subject to**, for every zone z:  Σ_z s − Σ_z d − net export_z = 0,  and −ATC ≤ flow ≤ ATC

The dual of each zone's balance is that **zone's price**. Note the conversion with `astype(float)` below: negating a boolean pandas column *inverts* it instead of making it −1 — a bug that silently produced nonsense prices while this lesson was being written.

In [3]:
def couple(atc):
    """Clear all zones together. atc: dict {(zone0, zone1): MW}. Returns prices, flows, welfare, accepted quantities."""
    borders = list(atc)
    n_o, n_b, n_l = len(offers), len(bids), len(borders)
    A = np.zeros((len(ZONES), n_o + n_b + n_l))
    for k, z in enumerate(ZONES):
        A[k, :n_o] = (offers.zone == z).astype(float)
        A[k, n_o:n_o + n_b] = -(bids.zone == z).astype(float)
        for j, (z0, z1) in enumerate(borders):
            A[k, n_o + n_b + j] = -1.0 if z == z0 else (1.0 if z == z1 else 0.0)   # flow z0 -> z1
    bounds = (list(zip(np.zeros(n_o), offers.qty)) + list(zip(np.zeros(n_b), bids.qty))
              + [(-atc[b], atc[b]) for b in borders])
    res = linprog(np.r_[offers.price, -bids.price, np.zeros(n_l)], A_eq=A, b_eq=np.zeros(len(ZONES)),
                  bounds=bounds, method="highs")
    return (pd.Series(res.eqlin.marginals, index=ZONES), pd.Series(res.x[n_o + n_b:], index=borders),
            -res.fun, res.x[:n_o], res.x[n_o:n_o + n_b])


def couple_pypsa(atc):
    """The same coupling in PyPSA: zones as buses, borders as bidirectional links."""
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", ZONES)
    for r in offers.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, marginal_cost=r.price)
    for r in bids.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, sign=-1, marginal_cost=-r.price)
    for (z0, z1), cap in atc.items():
        n.add("Link", f"{z0}-{z1}", bus0=z0, bus1=z1, p_nom=max(cap, 1e-6), p_min_pu=-1)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n.buses_t.marginal_price.iloc[0], -n.objective


scenarios = {
    "isolated": {("SI", "HR"): 0, ("HR", "HU"): 0, ("SI", "HU"): 0},
    "coupled, base ATC": {("SI", "HR"): 500, ("HR", "HU"): 400, ("SI", "HU"): 300},
    "coupled, large ATC": {("SI", "HR"): 900, ("HR", "HU"): 800, ("SI", "HU"): 500},
}
rows = {}
for name, atc in scenarios.items():
    prices, flows, welfare, _, _ = couple(atc)
    p_pypsa, w_pypsa = couple_pypsa(atc)
    assert np.allclose(prices.to_numpy(), p_pypsa.to_numpy()) and np.isclose(welfare, w_pypsa), "PyPSA agrees"
    rows[name] = {**{f"price_{z}": prices[z] for z in ZONES},
                  **{f"flow_{a}->{b}": f for (a, b), f in flows.items()}, "welfare_EUR_h": welfare}
coupling = pd.DataFrame(rows).T
coupling.round(1)

                    price_SI  price_HR  price_HU  flow_SI->HR  flow_HR->HU  \
isolated                95.0     160.0     100.0          0.0          0.0   
coupled, base ATC      100.0     120.0     100.0        500.0       -400.0   
coupled, large ATC     115.0     115.0     115.0        350.0       -800.0   

                    flow_SI->HU  welfare_EUR_h  
isolated                    0.0     21341700.0  
coupled, base ATC        -100.0     21375200.0  
coupled, large ATC        100.0     21379450.0  

Your LP and PyPSA agree in all three scenarios. Read the table from top to bottom:

- **Isolated**, each zone clears alone: Croatia must run its OCGT and pays **160 €/MWh**, while Slovenia (95) and Hungary (100) have spare cheap power.
- **Coupled with the base capacities**, cheap power flows into Croatia — 500 MW from Slovenia, 400 MW from Hungary, both borders **full** (a positive flow goes from the first zone to the second, so −400 on HR→HU means 400 MW from Hungary to Croatia). Croatia's price falls to 120 €/MWh, but stays above its neighbours' 100 €/MWh: the full borders keep the prices apart. This **price split** is the market's signal of congestion.
- **Coupled with larger capacities**, all three prices **converge** to one value, 115 €/MWh. The Croatian–Hungarian flow sits exactly at its 800 MW limit, but with equal prices on both sides that limit is worth nothing — its multiplier is zero (complementary slackness, Maths-2), so more capacity there would change nothing.

Welfare rises by 33,500 €/h from isolation to the base coupling — but only by 4,250 €/h more with the much larger capacities. The first megawatts of interconnection are worth far more than the last (Maths-3's falling value of line capacity).

### Congestion income: where the price difference goes

On a congested border, power is bought at the cheap side's price and sold at the expensive side's price. The difference times the flow is **congestion income** — it goes to the TSOs, and EU rules give priority to using it for keeping or increasing cross-border capacity. Together with consumer and producer surplus it adds up exactly to welfare:

In [4]:
atc = scenarios["coupled, base ATC"]
prices, flows, welfare, sold, bought = couple(atc)
p_off = offers.zone.map(prices).to_numpy()
p_bid = bids.zone.map(prices).to_numpy()
consumer = float(np.sum((bids.price.to_numpy() - p_bid) * bought))
producer = float(np.sum((p_off - offers.price.to_numpy()) * sold))
congestion = float(sum(f * (prices[b] - prices[a]) for (a, b), f in flows.items()))
assert np.isclose(consumer + producer + congestion, welfare), "welfare = CS + PS + congestion income"
print(f"consumer surplus {consumer:,.0f} + producer surplus {producer:,.0f} + congestion income {congestion:,.0f} "
      f"= {consumer + producer + congestion:,.0f} EUR/h = LP welfare")

consumer surplus 20,833,500 + producer surplus 523,700 + congestion income 18,000 = 21,375,200 EUR/h = LP welfare


Congestion income is 18,000 €/h: 500 MW × 20 €/MWh on the Slovenian border and 400 MW × 20 €/MWh on the Hungarian one. The Slovenian–Hungarian border carries power too, but between equal prices, so it earns nothing. This is the same identity as the nodal prices of Maths-3 — zones instead of nodes.

### Transfer capacities ignore physics — flow-based coupling

The model above treats each border like a pipe: power sent from Slovenia to Croatia flows on the Slovenia–Croatia border only. Real power flows obey Kirchhoff's laws (EE-1): it splits over **all** parallel paths in inverse proportion to their impedance. In a triangle of three equal lines, a transfer from Slovenia to Croatia goes two-thirds directly and one-third through Hungary:

In [5]:
# DC power flow on a triangle of equal reactances x: inject 1 MW at SI, withdraw at HR (EE-1.2 distribution factors)
x = 1.0
B = np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]) / x          # bus susceptance matrix, buses SI, HR, HU
theta = np.zeros(3)
theta[1:] = np.linalg.solve(B[1:, 1:], np.array([-1.0, 0.0]))      # SI is the reference; HR withdraws 1 MW
direct = (theta[0] - theta[1]) / x
via_hu = (theta[0] - theta[2]) / x
assert np.isclose(direct + via_hu, 1.0)
print(f"1 MW from SI to HR: {direct:.3f} MW on the direct line, {via_hu:.3f} MW loops through HU")

1 MW from SI to HR: 0.667 MW on the direct line, 0.333 MW loops through HU


A third of every Slovenia→Croatia trade loads the Hungarian lines — a **loop flow** the transfer-capacity model never sees. Fixed transfer capacities therefore have to be set conservatively, and they leave capacity unused. **Flow-based** market coupling, used in the Core region (which includes Croatia, Slovenia and Hungary) since June 2022, gives the algorithm the network's power transfer distribution factors (PTDFs) and the limits of critical branches, so every trade is charged for the lines it physically uses. You will compute PTDFs for real networks in Module 04.

### Real prices: Croatia, Slovenia, Hungary

With an ENTSO-E API token (M1.1) you can load the real day-ahead prices. Two practical points: prices come with a time-zone-aware index, and since October 2025 at 15-minute resolution — so convert everything to hourly averages before comparing zones. The download runs only when a token is present; **it was not executed while this course was built**, so check the numbers you get for plausibility (typically tens to a few hundred €/MWh, with occasional negative hours).

In [6]:
def load_day_ahead_prices(start, end, zones=("HR", "SI", "HU")):
    """Day-ahead prices (EUR/MWh) from ENTSO-E as hourly averages; requires ENTSOE_API_KEY in the environment."""
    from entsoe import EntsoePandasClient
    client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])
    start, end = pd.Timestamp(start, tz="Europe/Zagreb"), pd.Timestamp(end, tz="Europe/Zagreb")
    series = {z: client.query_day_ahead_prices(z, start=start, end=end) for z in zones}
    return pd.DataFrame({z: s.resample("h").mean() for z, s in series.items()})


def convergence_stats(prices, tol=0.01):
    """Share of hours with equal prices (within tol EUR/MWh) and mean absolute spread, for each pair of zones."""
    rows = {}
    zones = list(prices.columns)
    for i, a in enumerate(zones):
        for b in zones[i + 1:]:
            spread = (prices[a] - prices[b]).abs()
            rows[f"{a}-{b}"] = {"share_equal": float((spread <= tol).mean()), "mean_abs_spread": float(spread.mean())}
    return pd.DataFrame(rows).T


try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if os.environ.get("ENTSOE_API_KEY"):
    real = load_day_ahead_prices("2026-03-02", "2026-03-09")
    print(real.describe().round(1))
    print(convergence_stats(real).round(3))
else:
    print("No ENTSOE_API_KEY found - skipping the download. Add your token to .env (see M1.1) and re-run this cell.")

No ENTSOE_API_KEY found - skipping the download. Add your token to .env (see M1.1) and re-run this cell.


To make sure the analysis code itself works, run it on a small synthetic stand-in with the same shape as the real data — a 15-minute, time-zone-aware price series in which HR and SI are always equal, and SI and HU are equal in every other hour:

In [7]:
rng = np.random.default_rng(1)
idx15 = pd.date_range("2026-03-02", periods=4 * 24 * 7, freq="15min", tz="Europe/Zagreb")
hu = pd.Series(90 + 25 * np.sin(np.arange(len(idx15)) * 2 * np.pi / 96) + rng.normal(0, 5, len(idx15)), index=idx15)
si = hu + np.where(np.arange(len(idx15)) % 8 < 4, 0.0, 6.0)          # equal to HU in every other hour
stand_in = pd.DataFrame({"HR": si.copy(), "SI": si, "HU": hu})
hourly = stand_in.resample("h").mean()
stats = convergence_stats(hourly)
assert np.isclose(stats.loc["HR-SI", "share_equal"], 1.0) and np.isclose(stats.loc["SI-HU", "share_equal"], 0.5)
stats.round(3)

       share_equal  mean_abs_spread
HR-SI          1.0              0.0
HR-HU          0.5              3.0
SI-HU          0.5              3.0

The function reports what was built in: HR and SI always equal, SI and HU equal in half of the hours (after averaging the quarter-hours to hours), with a mean spread of 3 €/MWh. With real data, a high share of equal hours means the borders are rarely congested; persistent spreads point to congestion — and to the value of more interconnection, as in the coupling model above.

## Check your understanding

**Coupling**

1. In the base coupling, which border would you reinforce first, and how would you estimate the value of one more MW on it from the model's output?
2. Why can coupling *raise* the price in an exporting zone? Who in that zone gains and who loses?

**Design**

1. Why does intraday trading matter more as the share of wind and solar grows?
2. A transfer capacity is set to 500 MW, but the flow-based method would allow 650 MW for the same trades. Where does the difference come from, and why can flow-based coupling also *reduce* what is allowed on a border?